In [ ]:
import os
import sys
import pandas as pd

#avoid import errors
parent_dir=os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from scripts.CDS_machine import CDS_2_gffcomp
from scripts.statsCompression import gffstats_2data
from scripts.get_busco_db import species2lineage


In [ ]:
raw_data="../../data/species"
query_email=""
busco_database="../../data/busco_downloads/"

!ls $raw_data

# Run GeneID

In [ ]:
#geneid_command="time geneid -3P {parameter_file} {assembly} > {annotation.gff3}"

tr_sp=!ls $raw_data

for src in ["own", "git"]:
    #generate 1 file per source
    with open(f"../job_commands/SARpred_{src}.txt", "w") as out:
        #create predictions folder command
        pred_folder="../results/pred"
        print(f"mkdir -p {pred_folder}", file=out)
        for sp in tr_sp:
            
            #reference and model name are adaptable to the source
            ref=!realpath ../training_data/species/$sp/CLEAN*.fa
            model=f"{pred_folder}/{sp}_{src}.gff3"

            #parameter location changes depending on source
            param=!realpath ../results/trainedParams/$sp*.param
            param=param[0]
            #use github parameters
            if src=="git":
                filename_git=f"../training_data/geneid-parameter-files/parameter_files/{sp}*.param"

                try: #if no github parameters exist, dont runn geneID for git parameters

                    found_files=!realpath $filename_git 2>/dev/null #supress error output
                    #set up error for species with no github parameters
                    if "*" in found_files[0]: #found files is exactly filename_git if nothing is found
                        raise FileNotFoundError(f"There is no github parameters for this species")
                    
                except Exception as e:
                    print(f"{sp} presents error: {e}")
                    continue

                param=found_files[0]

            #geneid param_own/param_git fasta_assembly > gff_model
            command=f"time geneid -3P {param} {ref[0]} > {model}"

            #link inside the specie folder just in case
            logsDir_cmd=f"../results/specie_logs/{sp}/" #if running from exsting parameters create logs folder
            lnPred_cmd=f"ln -vf {model} ../results/specie_logs/{sp}/"

            print(command)
            print(command, file=out)
            print(logsDir_cmd, file=out)
            print(lnPred_cmd, file=out)
            

#Execute commands to predict CDS

# Prediction analysis

## Reference annotation(CDS) and Prediction comparison

In [ ]:

tr_sp=!ls $raw_data

for src in ["own", "git"]:
    with open(f"../job_commands/gffcomp_{src}.txt", "w") as f:
        
        #create folder to store comparison information
        resloc=f"../results/gffcmp"
        print(f"mkdir -p {resloc}", file=f)
        for sp in tr_sp:

            try:
                #if no github parameters exist, dont process this species git comparison
                filename_git=f"../training_data/geneid-parameter-files/parameter_files/{sp}*.param"
                found_files=!realpath $filename_git 2>/dev/null #supress error output

                if not found_files:
                    raise FileNotFoundError(f"There is no github parameters for this species")
                
                #reference annotation
                RefAnn=!realpath ../training_data/species/$sp/CDS_*gff
                RefAnn=RefAnn[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/pred/$sp*_$src*.gff3
                pred=pred[0]
                
                #adapt cds to be able to be compared
                adapted_RefAnn=f"../results/specie_logs/{sp}/CDSgff_{sp}.gff"
                
                CDS_2_gffcomp(RefAnn, adapted_RefAnn)

                #comparison command
                comparing_cmd=f"gffcompare -r {adapted_RefAnn} {pred} -o {resloc}/{sp}-cmp"
                
                #move all except stats, which is linked
                moveComparisons_cmd=f"mv {resloc}/!(*.stats*) ../results/specie_logs/{sp}/"
                lnStat_cmd=f"ln -vf {resloc}/{sp}-cmp.stats ../results/specie_logs/{sp}/"
                
                print(comparing_cmd, file=f)
                print(comparing_cmd)
                print(moveComparisons_cmd, file=f)
                print(lnStat_cmd, file=f)
                print(f'echo "done_{sp}"', file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue

In [ ]:
#Obtain all gffcmp results as 1 unified table
tr_sp=!ls $raw_data

statsData=[]
for sp in tr_sp:
    print(f"Species is {sp}")
    statsfile=f"../results/gffcmp/{sp}-cmp.stats"

    #return data for a concrete stats file
    parsed_dict=gffstats_2data(statsfile)
    #joind data from all stats files
    statsData.append(parsed_dict)

#transform data list to dataframe
df=pd.DataFrame(statsData)
df=df.fillna("NA")

df.to_csv("../results/gffcmp/specie_stats.csv", index=False, sep="\t")
print("Done")


### Result comparison

#### .stats files present info

**According to previous statistics**

- git and own have pretty similar results in both species, but in Pvivax they are closer(more similar) than in Pfalc. We can see this in the probability spreads, both are more disperse in Pfalc
- Individual distributuins show slight skew toward own data while transition matrix probabilities have a more stable distribution


**Both are confirmed by looking at the .stats files of the geneID results:**

- Git and Own results are pretty similar, specially in Pvivax
- Own results are slightly better


**Summary:** own produces better results

## BUSCO assessment

In [ ]:
#agat for gff>prot
tr_sp=!ls $raw_data
#tr_sp=["Drosophila_willistoni"]

#for src in ["git", "own"]:
for src in ["own"]:
    with open(f"../job_commands/buscoComp_{src}.txt", "w") as f:
        for sp in tr_sp:
            try:
                #if no github parameters exist, dont process this species git comparison
                filename_git=f"../training_data/geneid-parameter-files/parameter_files/{sp}*.param"
                found_files=!realpath $filename_git 2>/dev/null #supress error output

                if not found_files:
                    raise FileNotFoundError(f"There is no github parameters for this species")
                
                #reference sequence
                RefSeq=!realpath $raw_data/$sp/raw_*gn.f*a
                RefSeq=RefSeq[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/pred/$sp*_$src*.gff3
                pred=pred[0]

                #create folder to store outputs
                prot_res=f"../results/specie_logs/{sp}/{sp}_prot.fa" #protein sequence output
                busco_outPath=f"../results/busco/" #busco comparisons output
                busco_res=f"{sp}_busco" #busco out name

                #find lineage
                busco_lineage=species2lineage(sp, query_email, f"{busco_database}/file_versions.tsv", "odb12")

                #gff to protein command
                TOprotein_cmd=f"agat_sp_extract_sequences.pl \
                                -f {RefSeq} \
                                -g {pred} \
                                -t CDS \
                                -p \
                                -o {prot_res}"
                TOprotein_cmd=f"gffread -g {RefSeq} {pred} -y {prot_res}"

                print(TOprotein_cmd)
                print(TOprotein_cmd, file=f)
                
                busco_cmd=f"busco -m protein \
                            -i {prot_res} \
                            --download_path {busco_database} \
                            -l {busco_lineage} \
                            --cpu $(nproc) \
                            -f \
                            --opt-out-run-stats \
                            --out_path {busco_outPath} \
                            -o {busco_res}" #--offline #autolineage creates errors
                busco_cmd=busco_cmd.replace("                             ", " ")

                busco_plot=f"busco --opt-out-run-stats --plot {busco_outPath}{busco_res}"
                
                print(busco_cmd)
                print(busco_cmd, file=f)
                
                print(busco_plot)
                print(busco_plot, file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue
            

# Setup for IGV transcript comparison

In [ ]:
#generate symlinks to a igv allfiles folder
#s_results/specie/gff_*.gtf
#s_results/specie/gff_*.stats
#s_results/specie/*.gff3

    
#symlink files
#for raw files
clean_seq=!realpath ../data/species/**/CLEAN*.fa
clean_ann=!realpath ../data/species/**/CDS*ann.gff
rawData=clean_seq+clean_ann

for file in rawData:
    sp=file.split("/")[-2]
    #remove species shortened name from filename
    purename=file.split("/")[-1].split("_")
    purename="_".join([purename[0], purename[-1]])
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vf $file $dest/og_$purename


#for prediction result files
gff3=!realpath ../results/**/*.gff3

for file in gff3:
    sp=file.split("/")[-2]
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vf $file $dest


#for gffcompare files
gtfComp=!realpath ../results/**/gffcmp/*.gtf
statsComp=!realpath ../results/**/gffcmp/*.stats
gffComp=gtfComp+statsComp

for file in gffComp:
    sp=file.split("/")[-3]
    #name cleanup
    purename=file.split("/")[-1]
    purename=purename.replace(".annotated", "_ann").replace("GFF", "prediction")
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vf $file $dest/$purename




